# 05 · Calibration and honest evaluation

You now have two models that both claim to be about 89% accurate on `department`:
a ModernBERT encoder (lesson 03) and a fine-tuned Qwen3-0.6B decision model (lesson 04).
Both emit probabilities. The question this lesson answers is whether those probabilities
**mean anything**.

Often they do not. A softmax output is a ranking device: it is good at putting the right
answer first, and its sense of *how sure* it should be is a separate question — modern
networks are routinely described as saying 0.99 when they are right about 90% of the time.
If your plan is "auto-route anything above 0.95 confidence", that gap is the difference
between a working system and an incident report.

That is the textbook claim. **This lesson measures it rather than repeating it**, and what
we find is more interesting than the textbook — including the case where the standard fix
does not help.

This lesson is short and it is the one that gets a model deployed, because it produces
the only number a business can act on: **how much of the workload can we automate at an
acceptable error rate?**

**Time: about 10 minutes.**

In [1]:
import sys

sys.path.insert(0, "lib")
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import calibrate as C
import encoder as E
import readout as R
from tickets import QUESTIONS, generate_tickets, split

pd.set_option("display.width", 130)

tickets = generate_tickets(1200, seed=7)
parts = split(tickets, seed=7)
print(f"calibration split: {len(parts['calibration'])} tickets   "
      f"development split: {len(parts['development'])} tickets")

calibration split: 180 tickets   development split: 120 tickets


## 1. Load both models

The decision model comes from lesson 04's adapter; the encoder from lesson 03.

In [2]:
from mlx_lm import load

decision_model, decision_tok = load("Qwen/Qwen3-0.6B", adapter_path="adapters/decision")
encoder_model, encoder_tok = E.load("models/encoder")

print("decision model: Qwen3-0.6B + LoRA")
print("encoder:        ModernBERT-base, fine-tuned")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


decision model: Qwen3-0.6B + LoRA
encoder:        ModernBERT-base, fine-tuned


## 2. Are the probabilities honest?

Two measurements. The **reliability curve** buckets predictions by confidence and
compares each bucket's mean confidence with its actual accuracy — a perfectly calibrated
model lies on the diagonal. **ECE** collapses that curve into one number: the
population-weighted average gap.

We do this for both models on `department`, using the *development* split — the data we
are allowed to look at.

In [3]:
def probs_for_question(logits: np.ndarray) -> np.ndarray:
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=1, keepdims=True)


decision_dev_logits = {
    q: R.collect_logits(decision_model, decision_tok, parts["development"], q, QUESTIONS)
    for q in QUESTIONS
}
decision_gold = {q: R.gold_indices(parts["development"], q, QUESTIONS) for q in QUESTIONS}

encoder_report = E.evaluate(encoder_model, encoder_tok, parts["development"], verbose=False)

rows = []
for question in QUESTIONS:
    dp = probs_for_question(decision_dev_logits[question])
    dy = decision_gold[question]
    ep = encoder_report[question]["probs"]
    ey = encoder_report[question]["gold"]
    rows.append({
        "question": question,
        "decision acc": round(float((dp.argmax(1) == dy).mean()), 3),
        "decision ECE": round(C.expected_calibration_error(dp, dy), 3),
        "encoder acc": round(float((ep.argmax(1) == ey).mean()), 3),
        "encoder ECE": round(C.expected_calibration_error(ep, ey), 3),
    })
pd.DataFrame(rows)

,question,decision acc,decision ECE,encoder acc,encoder ECE
0,department,0.800,0.115,0.867,0.084
1,escalate,0.858,0.060,0.833,0.066
2,frustration,0.900,0.065,0.892,0.097


ECE around 0.05–0.10 means the model's confidence is off by five to ten points on
average. It says "90% sure" and is right about 80–85% of the time. That is not
catastrophic — and for a model trained with cross-entropy it is expected, because
cross-entropy rewards honest probabilities — but it is more than enough to break a
threshold you picked by intuition.

Let's look at the shape of the error rather than just its average:

In [4]:
dp = probs_for_question(decision_dev_logits["department"])
C.reliability_curve(dp, decision_gold["department"], n_bins=8)

,bin,count,mean confidence,accuracy,gap
0,0.00-0.12,0,NaN,NaN,NaN
1,0.12-0.25,0,NaN,NaN,NaN
2,0.25-0.38,1,0.351,0.000,-0.351
3,0.38-0.50,17,0.458,0.412,-0.046
4,0.50-0.62,13,0.559,0.462,-0.098
5,0.62-0.75,21,0.696,0.905,0.208
6,0.75-0.88,27,0.818,1.000,0.182
7,0.88-1.00,41,0.922,0.902,-0.020


Read the `gap` column: negative means overconfident (confidence above accuracy),
positive means underconfident. The bins with the most traffic are the ones that matter,
because they are where your decisions will actually be taken.

## 3. Temperature scaling

The fix is embarrassingly simple and it is the standard one. Instead of
`softmax(logits)`, use `softmax(logits / T)` for a single scalar `T`. It cannot change
which option wins — dividing by a positive number preserves order — so **accuracy is
untouched**. It only moves probability mass around.

`T` is fitted by minimising NLL on the **calibration split**, which is why lesson 02
insisted on a fourth split. Fitting it on development would make our reported calibration
optimistically biased, exactly like tuning hyperparameters on a test set.

In [5]:
temperatures, calibrated_rows = {}, []
for question in QUESTIONS:
    cal_logits = R.collect_logits(decision_model, decision_tok, parts["calibration"], question, QUESTIONS)
    cal_gold = R.gold_indices(parts["calibration"], question, QUESTIONS)

    T = C.fit_temperature(cal_logits, cal_gold)
    temperatures[question] = T

    raw = probs_for_question(decision_dev_logits[question])
    scaled = C.apply_temperature(decision_dev_logits[question], T)
    y = decision_gold[question]

    calibrated_rows.append({
        "question": question,
        "T": round(T, 3),
        "accuracy": round(float((scaled.argmax(1) == y).mean()), 3),
        "ECE before": round(C.expected_calibration_error(raw, y), 3),
        "ECE after": round(C.expected_calibration_error(scaled, y), 3),
        "NLL before": round(C.nll(raw, y), 3),
        "NLL after": round(C.nll(scaled, y), 3),
    })

calibration = pd.DataFrame(calibrated_rows)
calibration

,question,T,accuracy,ECE before,ECE after,NLL before,NLL after
0,department,1.428,0.800,0.115,0.165,0.679,0.693
1,escalate,1.307,0.858,0.060,0.103,0.416,0.437
2,frustration,0.923,0.900,0.065,0.061,0.365,0.366


**Read this table sceptically, because it does not say what the textbooks imply.**

1. **Accuracy is identical before and after.** It has to be — dividing logits by a
   positive constant cannot reorder them. Any tutorial showing accuracy change from
   temperature scaling is showing you a bug.
2. **On this data, temperature scaling made calibration *worse* on two of three
   questions — and made NLL worse too**, even though NLL is precisely what `T` was fitted
   to minimise.

That second point deserves a proper look, because it is not a bug in the code. Let's
measure where the temperature was fitted against where we are testing it:

In [6]:
print("Where does the fitted temperature actually help?\n")
for question in QUESTIONS:
    cal_logits = R.collect_logits(decision_model, decision_tok, parts["calibration"], question, QUESTIONS)
    cal_gold = R.gold_indices(parts["calibration"], question, QUESTIONS)
    T = temperatures[question]

    def nll_at(logits, y, t):
        return C.nll(C.apply_temperature(logits, t), y)

    cal_before, cal_after = nll_at(cal_logits, cal_gold, 1.0), nll_at(cal_logits, cal_gold, T)
    dev_before = nll_at(decision_dev_logits[question], decision_gold[question], 1.0)
    dev_after = nll_at(decision_dev_logits[question], decision_gold[question], T)

    print(f"{question:12s} T={T:.3f}")
    print(f"   calibration split (T was fitted here):  NLL {cal_before:.4f} -> {cal_after:.4f}  "
          f"{'improved' if cal_after < cal_before else 'worse'}")
    print(f"   development split (we measure here):     NLL {dev_before:.4f} -> {dev_after:.4f}  "
          f"{'improved' if dev_after < dev_before else 'worse'}")

Where does the fitted temperature actually help?



department   T=1.428
   calibration split (T was fitted here):  NLL 0.8529 -> 0.8155  improved
   development split (we measure here):     NLL 0.6791 -> 0.6930  worse


escalate     T=1.307
   calibration split (T was fitted here):  NLL 0.5458 -> 0.5352  improved
   development split (we measure here):     NLL 0.4163 -> 0.4371  worse


frustration  T=0.923
   calibration split (T was fitted here):  NLL 0.3276 -> 0.3258  improved
   development split (we measure here):     NLL 0.3654 -> 0.3662  worse


The optimizer is doing its job: **on the split it was fitted to, NLL improves every
time.** The problem is that this is a *different split* from the one we measure on.

The mechanism is visible in the accuracies. Our calibration split happens to be a
**harder sample** than development — about 0.75 against 0.80 on `department`. A model that
is less accurate ought to be less confident, so the optimizer asks for a softer
distribution (`T = 1.43`); development, being easier, would prefer something nearer 1.2.
The fitted value is right for the data it saw, and slightly wrong for the data it did not.

Three things to take from this:

1. **Calibration parameters are fitted parameters.** A few hundred calibration rows is
   marginal; a few thousand is comfortable. Fit and *validate* on different splits, or
   you are reporting the fit's optimism rather than its transfer.
2. **A model trained with cross-entropy is often already close to calibrated.** Softmax
   outputs are overconfident in general, but this model's ECE was 0.06–0.12 before any
   correction. There was little for a single scalar to fix, and the noise in estimating it
   was larger than the error it removes.
3. **Where temperature scaling earns its reputation is on models that are *badly*
   miscalibrated** — one trained on very few examples, or one whose training pushed
   confidence up without pushing accuracy with it. There `T` comes out well above 1 and
   the improvement is large. One measurement tells you which regime you are in.

So the honest summary is not "temperature scaling fixes confidence". It is: **temperature
scaling is one fitted scalar, it is not free, and you must measure whether it helped.**

In [7]:
# The reliability curve after scaling, for the question that improved least.
question = "department"
scaled = C.apply_temperature(decision_dev_logits[question], temperatures[question])
print(f"{question}: T = {temperatures[question]:.3f}\n")
print("before:")
print(C.reliability_curve(probs_for_question(decision_dev_logits[question]), decision_gold[question], n_bins=6).to_string(index=False))
print("\nafter:")
print(C.reliability_curve(scaled, decision_gold[question], n_bins=6).to_string(index=False))

department: T = 1.428

before:
      bin  count  mean confidence  accuracy    gap
0.00-0.17      0              NaN       NaN    NaN
0.17-0.33      0              NaN       NaN    NaN
0.33-0.50     18            0.452     0.389 -0.063
0.50-0.67     18            0.585     0.556 -0.030
0.67-0.83     32            0.750     0.969  0.219
0.83-1.00     52            0.909     0.923  0.014

after:
      bin  count  mean confidence  accuracy    gap
0.00-0.17      0              NaN       NaN    NaN
0.17-0.33      1            0.319     0.000 -0.319
0.33-0.50     24            0.431     0.458  0.028
0.50-0.67     30            0.590     0.800  0.210
0.67-0.83     50            0.761     0.920  0.159
0.83-1.00     15            0.887     1.000  0.113


## 4. The number that gets a model deployed

Accuracy is not actionable. "89% accurate" does not tell anyone what to do. The
actionable framing is **selective prediction**: the model may abstain, a human handles
what it declines, and you ask how much of the queue can be cleared automatically while
keeping the error rate among *accepted* items under a budget.

That is what `coverage_at_error` computes: sort by confidence, accept the most confident
first, and find the largest set you can accept while staying under budget.

In [8]:
rows = []
for question in QUESTIONS:
    scaled = C.apply_temperature(decision_dev_logits[question], temperatures[question])
    y = decision_gold[question]
    for budget in [0.02, 0.05, 0.10]:
        cov = C.coverage_at_error(scaled, y, target_error=budget)
        rows.append({
            "question": question,
            "error budget": f"{budget:.0%}",
            "coverage": f"{cov['coverage']:.1%}",
            "threshold to apply": round(cov["threshold"], 3),
            "actual error": f"{cov['error_rate']:.1%}",
        })
pd.DataFrame(rows)

,question,error budget,coverage,threshold to apply,actual error
0,department,2%,14.2%,0.830,0.0%
1,department,5%,67.5%,0.592,4.9%
2,department,10%,77.5%,0.515,9.7%
3,escalate,2%,10.0%,0.860,0.0%
4,escalate,5%,10.0%,0.860,0.0%
5,escalate,10%,75.8%,0.703,9.9%
6,frustration,2%,11.7%,0.978,0.0%
7,frustration,5%,11.7%,0.978,0.0%
8,frustration,10%,100.0%,0.497,10.0%


This is the sentence you take to the business: *"At a 5% error budget we can auto-route
X% of tickets, by acting on every prediction the model is at least 0.87 confident about."*
That is a decision someone can make. "89% accurate" is not.

Two warnings:

* **These numbers come from 120 development tickets, and from a temperature we just showed
  does not transfer well.** A coverage estimate from 120 rows is noisy, and the threshold
  it implies is noisier still. On a real workload you would use a few thousand rows for
  both, and re-check that the fitted temperature actually helps the split you are
  reporting on.
* **The error budget must come from the business, not from the model.** A wrong refund
  routing and a wrong tone-of-voice score do not have the same cost. One model, several
  budgets, several thresholds.

In [9]:
C.coverage_curve(
    C.apply_temperature(decision_dev_logits["escalate"], temperatures["escalate"]),
    decision_gold["escalate"],
    steps=12,
)

,coverage,threshold,error rate
0,0.050,0.871,0.0000
1,0.133,0.848,0.1250
2,0.217,0.822,0.1154
3,0.308,0.800,0.1081
4,0.392,0.776,0.1064
5,0.475,0.776,0.1053
6,0.567,0.759,0.1029
7,0.650,0.741,0.1026
8,0.733,0.722,0.0909
9,0.825,0.683,0.1313


## 5. Is the difference between the two models real?

Lesson 03 left an open question: the encoder and the decision model have similar accuracy.
Is the encoder actually better, or is the gap smaller than the noise in a 120-row
evaluation?

The right tool is a **paired bootstrap**. Resample the development rows with replacement,
recompute the difference in accuracy each time, and look at the spread. Pairing matters:
both models saw the same rows, so we resample rows rather than models.

In [10]:
rows = []
for question in QUESTIONS:
    decision_pred = C.apply_temperature(
        decision_dev_logits[question], temperatures[question]
    ).argmax(axis=1)
    encoder_pred = encoder_report[question]["probs"].argmax(axis=1)
    y = decision_gold[question]

    # align the encoder's label order to the readout's
    assert list(encoder_report[question]["classes"]) == list(
        QUESTIONS[question].get("options") or ["false", "true"]
    ), question

    result = C.paired_bootstrap(encoder_pred, decision_pred, y)
    rows.append({
        "question": question,
        "encoder - decision": round(result["difference"], 3),
        "95% CI": f"[{result['ci_low']:+.3f}, {result['ci_high']:+.3f}]",
        "real difference?": "yes" if result["significant"] else "no — CI spans zero",
    })
pd.DataFrame(rows)

,question,encoder - decision,95% CI,real difference?
0,department,0.067,"[+0.000, +0.142]",no — CI spans zero
1,escalate,-0.025,"[-0.100, +0.042]",no — CI spans zero
2,frustration,-0.008,"[-0.033, +0.017]",no — CI spans zero


**A confidence interval that spans zero means you have not measured a difference**, no
matter how large the point estimate looks. This is the discipline that stops you from
shipping a change because it gained 0.4 points on a 120-row evaluation.

Kev's tooling bakes the same idea in from the other direction: its `plan_size.py` takes a
target effect ("detect a +5 point gain at 80% power") and tells you how many labelled
records you need before running anything. Its author's measured result on a real workload
is worth remembering: 400 records gave **+0.6 points with a ±6 point interval** — a
measurement that cannot distinguish a real gain from nothing — while 1050 records gave
**+5.9 points with a [+2.3, +9.7] interval**, which can. Fine-tuning gains are usually
real but modest, and you need enough data to see them at all.

## 6. Three ways to fool yourself

**Fit the temperature on the data you report.** The most common. If you tune `T` on
development and then quote ECE on development, your calibration number is optimistic by
however much the fit absorbed. Use the calibration split.

**Quote ECE without its bin count.** ECE depends on binning. With two bins it is almost
always small; with 50 bins it is almost always large. Report the bin count, and show the
reliability curve so the reader can see the shape.

**Assume calibration transfers.** `T` is fitted to a distribution. Change the traffic —
a new product line, a different customer segment, tickets in a new language — and the
calibration is fitted to the old one. This is why the notebook keeps both the leaky and
the template-held-out development sets: they are two different distributions, and you can
watch the same `T` behave differently on each. Re-fit calibration whenever the input
distribution moves, and monitor ECE in production where you can.

---

### Exercises

1. **Break the temperature.** Fit `T` on `development` instead of `calibration` and
   compare the resulting ECE on `development`. How much optimism did you buy? Repeat
   a few times with different splits to see the size of the effect.

2. **Vary the bins.** Compute ECE for the same predictions with 5, 15 and 40 bins. How
   much does the number move? Now you know why a bare ECE figure is not comparable across
   papers unless the binning matches.

3. **Calibrate the encoder too.** The encoder is already reasonably calibrated (its
   cross-entropy training saw to that). Fit a `T` anyway and compare the ECE improvement
   against the decision model's. Which one needed help, and does that match your
   expectation about which model was overfitting?

4. **Cost-weighted thresholds.** Rework section 4 so a wrong `escalate=true` costs 10x a
   wrong `escalate=false` (a human review is expensive). What error budget does that
   imply, and what coverage do you get? This is the analysis that turns a model into a
   decision.

---
**Next:** [06 · Serving it](06_serving.ipynb) — putting this behind an HTTP endpoint.